In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Performance Visualization\n",
    "## Screening Performance Metrics and Visualizations\n",
    "\n",
    "This notebook creates comprehensive performance visualizations for systematic review screening."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "sys.path.append('../src')\n",
    "\n",
    "from performance import PerformanceCalculator\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import plotly.graph_objects as go\n",
    "import plotly.express as px\n",
    "from plotly.subplots import make_subplots\n",
    "import json\n",
    "import yaml\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set styles\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "sns.set_palette(\"Set2\")\n",
    "\n",
    "# Display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "\n",
    "print(\"✅ Libraries imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Performance Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Configuration\n",
    "REVIEWS = {\n",
    "    'insulin': {\n",
    "        'python_csv': '../data/output/insulin_screening/screening_decisions_insulin.csv',\n",
    "        'covidence_total': '../data/insulin/covidence_total.csv',\n",
    "        'config': '../config/insulin_config.yaml',\n",
    "        'python_time': 9,  # minutes\n",
    "        'manual_time': 96  # hours\n",
    "    },\n",
    "    'glp1': {\n",
    "        'python_csv': '../data/output/glp1_screening/screening_decisions_glp1.csv',\n",
    "        'covidence_total': '../data/glp1/covidence_total.csv',\n",
    "        'config': '../config/glp1_config.yaml',\n",
    "        'python_time': 12,  # minutes\n",
    "        'manual_time': 720  # hours\n",
    "    }\n",
    "}\n",
    "\n",
    "# Select reviews to analyze\n",
    "selected_reviews = ['insulin', 'glp1']  # Analyze both\n",
    "\n",
    "print(\"📊 Selected Reviews for Performance Analysis:\")\n",
    "for review in selected_reviews:\n",
    "    print(f\"  • {review.upper()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Calculate performance for each review\n",
    "all_metrics = {}\n",
    "\n",
    "for review in selected_reviews:\n",
    "    print(f\"\\n🔍 Calculating performance for: {review.upper()}\")\n",
    "    \n",
    "    config = REVIEWS[review]\n",
    "    \n",
    "    # Initialize calculator\n",
    "    calculator = PerformanceCalculator()\n",
    "    \n",
    "    try:\n",
    "        # Calculate performance\n",
    "        metrics = calculator.calculate(config['python_csv'], config['covidence_total'])\n",
    "        \n",
    "        # Add time efficiency\n",
    "        if config['python_time'] and config['manual_time']:\n",
    "            manual_minutes = config['manual_time'] * 60\n",
    "            time_saved = manual_minutes - config['python_time']\n",
    "            percent_saved = time_saved / manual_minutes if manual_minutes > 0 else 0\n",
    "            speed_increase = manual_minutes / config['python_time'] if config['python_time'] > 0 else 0\n",
    "            \n",
    "            metrics.update({\n",
    "                'python_time_minutes': config['python_time'],\n",
    "                'manual_time_hours': config['manual_time'],\n",
    "                'manual_time_minutes': manual_minutes,\n",
    "                'time_saved_minutes': time_saved,\n",
    "                'time_saved_hours': time_saved / 60,\n",
    "                'percent_time_saved': percent_saved,\n",
    "                'speed_increase': speed_increase\n",
    "            })\n",
    "        \n",
    "        all_metrics[review] = metrics\n",
    "        print(f\"  ✓ Calculated {len(metrics)} metrics\")\n",
    "        \n",
    "    except Exception as e:\n",
    "        print(f\"  ✗ Error: {e}\")\n",
    "        all_metrics[review] = {}\n",
    "\n",
    "print(f\"\\n✅ Performance calculated for {len([m for m in all_metrics.values() if m])} reviews\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Create Comparative Visualizations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create comprehensive dashboard\n",
    "print(\"📈 Creating performance dashboard...\")\n",
    "\n",
    "fig = plt.figure(figsize=(18, 12))\n",
    "fig.suptitle('Systematic Review Screening Performance Dashboard', fontsize=16, fontweight='bold', y=0.98)\n",
    "\n",
    "# Define grid\n",
    "gs = fig.add_gridspec(3, 4, hspace=0.4, wspace=0.3)\n",
    "\n",
    "# 1. Core Metrics Comparison\n",
    "ax1 = fig.add_subplot(gs[0, :2])\n",
    "\n",
    "core_metrics = ['sensitivity', 'specificity', 'precision', 'accuracy']\n",
    "x = np.arange(len(core_metrics))\n",
    "width = 0.35\n",
    "\n",
    "for i, review in enumerate(selected_reviews):\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        values = [all_metrics[review].get(metric, 0) for metric in core_metrics]\n",
    "        offset = width * i - width/2\n",
    "        bars = ax1.bar(x + offset, values, width, label=review.upper(), \n",
    "                      alpha=0.8, edgecolor='black')\n",
    "        \n",
    "        # Add value labels\n",
    "        for bar, value in zip(bars, values):\n",
    "            height = bar.get_height()\n",
    "            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,\n",
    "                    f'{value:.1%}', ha='center', va='bottom', fontsize=9)\n",
    "\n",
    "ax1.set_xticks(x)\n",
    "ax1.set_xticklabels(['Sensitivity', 'Specificity', 'Precision', 'Accuracy'])\n",
    "ax1.set_ylim(0, 1.1)\n",
    "ax1.set_ylabel('Score')\n",
    "ax1.set_title('Core Performance Metrics', fontsize=14, fontweight='bold')\n",
    "ax1.legend()\n",
    "ax1.grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "# 2. Confusion Matrix Comparison\n",
    "ax2 = fig.add_subplot(gs[0, 2:])\n",
    "\n",
    "# Calculate totals\n",
    "confusion_data = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        m = all_metrics[review]\n",
    "        if all(k in m for k in ['true_positives', 'false_positives', 'false_negatives', 'true_negatives']):\n",
    "            confusion_data.append({\n",
    "                'Review': review.upper(),\n",
    "                'TP': m['true_positives'],\n",
    "                'FP': m['false_positives'],\n",
    "                'FN': m['false_negatives'],\n",
    "                'TN': m['true_negatives'],\n",
    "                'Total': m.get('total_matched', 0)\n",
    "            })\n",
    "\n",
    "if confusion_data:\n",
    "    confusion_df = pd.DataFrame(confusion_data)\n",
    "    x_conf = np.arange(len(confusion_df))\n",
    "    width_conf = 0.2\n",
    "    \n",
    "    for i, (col, label) in enumerate([('TP', 'True Positives'), ('FP', 'False Positives'), \n",
    "                                       ('FN', 'False Negatives'), ('TN', 'True Negatives')]):\n",
    "        values = confusion_df[col].values if col in confusion_df.columns else [0]*len(confusion_df)\n",
    "        ax2.bar(x_conf + i*width_conf - width_conf*1.5, values, width_conf, \n",
    "               label=label, alpha=0.8)\n",
    "    \n",
    "    ax2.set_xticks(x_conf)\n",
    "    ax2.set_xticklabels(confusion_df['Review'].values)\n",
    "    ax2.set_ylabel('Count')\n",
    "    ax2.set_title('Confusion Matrix Comparison', fontsize=14, fontweight='bold')\n",
    "    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')\n",
    "    ax2.grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "# 3. Time Efficiency\n",
    "ax3 = fig.add_subplot(gs[1, :2])\n",
    "\n",
    "time_data = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        m = all_metrics[review]\n",
    "        if 'speed_increase' in m:\n",
    "            time_data.append({\n",
    "                'Review': review.upper(),\n",
    "                'Python (min)': m.get('python_time_minutes', 0),\n",
    "                'Manual (hours)': m.get('manual_time_hours', 0),\n",
    "                'Speed Increase': m['speed_increase'],\n",
    "                'Time Saved (hours)': m.get('time_saved_hours', 0)\n",
    "            })\n",
    "\n",
    "if time_data:\n",
    "    time_df = pd.DataFrame(time_data)\n",
    "    \n",
    "    # Time comparison\n",
    "    x_time = np.arange(len(time_df))\n",
    "    width_time = 0.35\n",
    "    \n",
    "    python_times = time_df['Python (min)'].values / 60  # Convert to hours for comparison\n",
    "    manual_times = time_df['Manual (hours)'].values\n",
    "    \n",
    "    bars1 = ax3.bar(x_time - width_time/2, python_times, width_time, \n",
    "                   label='Python (hours)', color='#2ecc71', alpha=0.8)\n",
    "    bars2 = ax3.bar(x_time + width_time/2, manual_times, width_time, \n",
    "                   label='Manual (hours)', color='#e74c3c', alpha=0.8)\n",
    "    \n",
    "    ax3.set_xticks(x_time)\n",
    "    ax3.set_xticklabels(time_df['Review'].values)\n",
    "    ax3.set_ylabel('Time (hours)')\n",
    "    ax3.set_title('Time Efficiency Comparison', fontsize=14, fontweight='bold')\n",
    "    ax3.legend()\n",
    "    ax3.grid(True, alpha=0.3, axis='y')\n",
    "    \n",
    "    # Add time saved annotation\n",
    "    for i, (python_hours, manual_hours) in enumerate(zip(python_times, manual_times)):\n",
    "        time_saved = manual_hours - python_hours\n",
    "        ax3.text(i, max(python_hours, manual_hours) + 0.5,\n",
    "                f'Saved: {time_saved:.1f}h', ha='center', fontsize=9)\n",
    "\n",
    "# 4. Speed Increase\n",
    "ax4 = fig.add_subplot(gs[1, 2])\n",
    "\n",
    "if time_data:\n",
    "    speed_increases = time_df['Speed Increase'].values\n",
    "    reviews = time_df['Review'].values\n",
    "    colors = ['#3498db', '#9b59b6', '#1abc9c']\n",
    "    \n",
    "    bars = ax4.bar(reviews, speed_increases, color=colors[:len(reviews)], alpha=0.8, edgecolor='black')\n",
    "    ax4.set_ylabel('Speed Increase (x)')\n",
    "    ax4.set_title('Speed Increase Factor', fontsize=14, fontweight='bold')\n",
    "    ax4.grid(True, alpha=0.3, axis='y')\n",
    "    \n",
    "    # Add labels\n",
    "    for bar, speed in zip(bars, speed_increases):\n",
    "        height = bar.get_height()\n",
    "        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.5,\n",
    "                f'{speed:.1f}x', ha='center', fontsize=10, fontweight='bold')\n",
    "\n",
    "# 5. Workload Reduction\n",
    "ax5 = fig.add_subplot(gs[1, 3])\n",
    "\n",
    "# Calculate workload reduction\n",
    "workload_data = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        m = all_metrics[review]\n",
    "        if 'true_negatives' in m and 'total_matched' in m:\n",
    "            workload_reduction = m['true_negatives'] / m['total_matched'] if m['total_matched'] > 0 else 0\n",
    "            workload_data.append({\n",
    "                'Review': review.upper(),\n",
    "                'Workload Reduction': workload_reduction\n",
    "            })\n",
    "\n",
    "if workload_data:\n",
    "    workload_df = pd.DataFrame(workload_data)\n",
    "    \n",
    "    bars = ax5.bar(workload_df['Review'], workload_df['Workload Reduction'], \n",
    "                  color=['#e67e22', '#16a085'], alpha=0.8, edgecolor='black')\n",
    "    ax5.set_ylim(0, 1)\n",
    "    ax5.set_ylabel('Workload Reduction')\n",
    "    ax5.set_title('Workload Reduction', fontsize=14, fontweight='bold')\n",
    "    ax5.grid(True, alpha=0.3, axis='y')\n",
    "    \n",
    "    # Add percentage labels\n",
    "    for bar, reduction in zip(bars, workload_df['Workload Reduction']):\n",
    "        height = bar.get_height()\n",
    "        ax5.text(bar.get_x() + bar.get_width()/2., height + 0.02,\n",
    "                f'{reduction:.1%}', ha='center', fontsize=10, fontweight='bold')\n",
    "\n",
    "# 6. Summary Statistics\n",
    "ax6 = fig.add_subplot(gs[2, :])\n",
    "ax6.axis('off')\n",
    "\n",
    "# Create summary table\n",
    "summary_rows = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        m = all_metrics[review]\n",
    "        summary_rows.append([\n",
    "            review.upper(),\n",
    "            f\"{m.get('sensitivity', 0):.1%}\",\n",
    "            f\"{m.get('specificity', 0):.1%}\",\n",
    "            f\"{m.get('f1_score', 0):.3f}\",\n",
    "            f\"{m.get('accuracy', 0):.1%}\",\n",
    "            f\"{m.get('speed_increase', 0):.1f}x\" if 'speed_increase' in m else 'N/A',\n",
    "            f\"{m.get('time_saved_hours', 0):.1f}h\" if 'time_saved_hours' in m else 'N/A',\n",
    "            f\"{m.get('total_matched', 0):,}\"\n",
    "        ])\n",
    "\n",
    "if summary_rows:\n",
    "    headers = ['Review', 'Sensitivity', 'Specificity', 'F1 Score', 'Accuracy', \n",
    "               'Speed Increase', 'Time Saved', 'Matched Studies']\n",
    "    \n",
    "    # Create table\n",
    "    table = ax6.table(cellText=summary_rows, colLabels=headers, \n",
    "                     cellLoc='center', loc='center',\n",
    "                     colColours=['#3498db']*len(headers),\n",
    "                     cellColours=[['#f2f2f2']*len(headers) for _ in summary_rows])\n",
    "    \n",
    "    table.auto_set_font_size(False)\n",
    "    table.set_fontsize(10)\n",
    "    table.scale(1, 1.5)\n",
    "    \n",
    "    ax6.set_title('Performance Summary', fontsize=14, fontweight='bold', y=0.95)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Interactive Visualizations with Plotly"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create interactive dashboard with Plotly\n",
    "print(\"📊 Creating interactive dashboard...\")\n",
    "\n",
    "# Prepare data for plotly\n",
    "plotly_data = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        m = all_metrics[review]\n",
    "        plotly_data.append({\n",
    "            'Review': review.upper(),\n",
    "            'Sensitivity': m.get('sensitivity', 0) * 100,\n",
    "            'Specificity': m.get('specificity', 0) * 100,\n",
    "            'Precision': m.get('precision', 0) * 100,\n",
    "            'Accuracy': m.get('accuracy', 0) * 100,\n",
    "            'F1_Score': m.get('f1_score', 0) * 100,\n",
    "            'Speed_Increase': m.get('speed_increase', 0),\n",
    "            'Time_Saved_Hours': m.get('time_saved_hours', 0),\n",
    "            'Workload_Reduction': m.get('true_negatives', 0) / m.get('total_matched', 1) * 100 if 'total_matched' in m else 0\n",
    "        })\n",
    "\n",
    "if plotly_data:\n",
    "    df_plotly = pd.DataFrame(plotly_data)\n",
    "    \n",
    "    # Create subplots\n",
    "    fig_plotly = make_subplots(\n",
    "        rows=2, cols=3,\n",
    "        subplot_titles=('Performance Metrics', 'Time Efficiency', 'Confusion Matrix',\n",
    "                       'Speed Comparison', 'Workload Reduction', 'Summary'),\n",
    "        specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'heatmap'}],\n",
    "               [{'type': 'bar'}, {'type': 'bar'}, {'type': 'table'}]]\n",
    "    )\n",
    "    \n",
    "    # 1. Performance Metrics\n",
    "    metrics = ['Sensitivity', 'Specificity', 'Precision', 'Accuracy']\n",
    "    for review in df_plotly['Review'].unique():\n",
    "        review_data = df_plotly[df_plotly['Review'] == review]\n",
    "        fig_plotly.add_trace(\n",
    "            go.Bar(name=review, x=metrics, \n",
    "                  y=[review_data[m].values[0] for m in ['Sensitivity', 'Specificity', 'Precision', 'Accuracy']],\n",
    "                  text=[f'{val:.1f}%' for val in [review_data[m].values[0] for m in ['Sensitivity', 'Specificity', 'Precision', 'Accuracy']]],\n",
    "                  textposition='auto'),\n",
    "            row=1, col=1\n",
    "        )\n",
    "    \n",
    "    # 2. Time Efficiency\n",
    "    for review in df_plotly['Review'].unique():\n",
    "        review_data = df_plotly[df_plotly['Review'] == review]\n",
    "        fig_plotly.add_trace(\n",
    "            go.Bar(name=review, x=['Time Saved'], \n",
    "                  y=[review_data['Time_Saved_Hours'].values[0]],\n",
    "                  text=[f'{review_data[\"Time_Saved_Hours\"].values[0]:.1f}h'],\n",
    "                  textposition='auto'),\n",
    "            row=1, col=2\n",
    "        )\n",
    "    \n",
    "    # 3. Confusion Matrix Heatmap\n",
    "    confusion_values = []\n",
    "    for review in selected_reviews:\n",
    "        if review in all_metrics and all_metrics[review]:\n",
    "            m = all_metrics[review]\n",
    "            if all(k in m for k in ['true_positives', 'false_positives', 'false_negatives', 'true_negatives']):\n",
    "                confusion_values.append([\n",
    "                    m['true_positives'], m['false_positives'],\n",
    "                    m['false_negatives'], m['true_negatives']\n",
    "                ])\n",
    "    \n",
    "    if confusion_values:\n",
    "        fig_plotly.add_trace(\n",
    "            go.Heatmap(z=confusion_values,\n",
    "                      x=['TP', 'FP', 'FN', 'TN'],\n",
    "                      y=[r.upper() for r in selected_reviews if r in all_metrics],\n",
    "                      text=confusion_values,\n",
    "                      texttemplate='%{text:,}',\n",
    "                      colorscale='Viridis'),\n",
    "            row=1, col=3\n",
    "        )\n",
    "    \n",
    "    # 4. Speed Comparison\n",
    "    fig_plotly.add_trace(\n",
    "        go.Bar(x=df_plotly['Review'], y=df_plotly['Speed_Increase'],\n",
    "              text=[f'{x:.1f}x' for x in df_plotly['Speed_Increase']],\n",
    "              textposition='auto',\n",
    "              marker_color=['#2ecc71', '#3498db', '#9b59b6']),\n",
    "        row=2, col=1\n",
    "    )\n",
    "    \n",
    "    # 5. Workload Reduction\n",
    "    fig_plotly.add_trace(\n",
    "        go.Bar(x=df_plotly['Review'], y=df_plotly['Workload_Reduction'],\n",
    "              text=[f'{x:.1f}%' for x in df_plotly['Workload_Reduction']],\n",
    "              textposition='auto',\n",
    "              marker_color=['#e74c3c', '#f39c12', '#1abc9c']),\n",
    "        row=2, col=2\n",
    "    )\n",
    "    \n",
    "    # 6. Summary Table\n",
    "    summary_data = []\n",
    "    for review in selected_reviews:\n",
    "        if review in all_metrics and all_metrics[review]:\n",
    "            m = all_metrics[review]\n",
    "            summary_data.append([\n",
    "                review.upper(),\n",
    "                f\"{m.get('sensitivity', 0):.1%}\",\n",
    "                f\"{m.get('specificity', 0):.1%}\",\n",
    "                f\"{m.get('f1_score', 0):.3f}\",\n",
    "                f\"{m.get('speed_increase', 0):.1f}x\" if 'speed_increase' in m else 'N/A',\n",
    "                f\"{m.get('time_saved_hours', 0):.1f}h\" if 'time_saved_hours' in m else 'N/A'\n",
    "            ])\n",
    "    \n",
    "    fig_plotly.add_trace(\n",
    "        go.Table(\n",
    "            header=dict(values=['Review', 'Sensitivity', 'Specificity', 'F1 Score', 'Speed Increase', 'Time Saved'],\n",
    "                       fill_color='#3498db',\n",
    "                       font=dict(color='white', size=12)),\n",
    "            cells=dict(values=list(zip(*summary_data)),\n",
    "                      fill_color='#f2f2f2',\n",
    "                      font=dict(size=11))\n",
    "        ),\n",
    "        row=2, col=3\n",
    "    )\n",
    "    \n",
    "    # Update layout\n",
    "    fig_plotly.update_layout(\n",
    "        height=1000,\n",
    "        showlegend=True,\n",
    "        title_text=\"Interactive Performance Dashboard\",\n",
    "        title_x=0.5\n",
    "    )\n",
    "    \n",
    "    fig_plotly.show()\n",
    "    print(\"✅ Interactive dashboard created!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Export Performance Reports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save comprehensive performance report\n",
    "print(\"💾 Saving performance reports...\")\n",
    "\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        output_dir = Path(f'../data/output/{review}_performance')\n",
    "        output_dir.mkdir(parents=True, exist_ok=True)\n",
    "        \n",
    "        # Save metrics\n",
    "        metrics_file = output_dir / 'performance_metrics.json'\n",
    "        with open(metrics_file, 'w') as f:\n",
    "            json.dump(all_metrics[review], f, indent=2)\n",
    "        \n",
    "        # Create summary report\n",
    "        summary = {\n",
    "            'review': review.upper(),\n",
    "            'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),\n",
    "            'performance_summary': {\n",
    "                'sensitivity': f\"{all_metrics[review].get('sensitivity', 0):.1%}\",\n",
    "                'specificity': f\"{all_metrics[review].get('specificity', 0):.1%}\",\n",
    "                'precision': f\"{all_metrics[review].get('precision', 0):.1%}\",\n",
    "                'accuracy': f\"{all_metrics[review].get('accuracy', 0):.1%}\",\n",
    "                'f1_score': f\"{all_metrics[review].get('f1_score', 0):.3f}\"\n",
    "            },\n",
    "            'efficiency_summary': {\n",
    "                'python_time': f\"{all_metrics[review].get('python_time_minutes', 0)} minutes\",\n",
    "                'manual_time': f\"{all_metrics[review].get('manual_time_hours', 0)} hours\",\n",
    "                'time_saved': f\"{all_metrics[review].get('time_saved_hours', 0):.1f} hours\",\n",
    "                'speed_increase': f\"{all_metrics[review].get('speed_increase', 0):.1f}x\",\n",
    "                'percent_time_saved': f\"{all_metrics[review].get('percent_time_saved', 0):.1%}\"\n",
    "            },\n",
    "            'confusion_matrix': {\n",
    "                'true_positives': all_metrics[review].get('true_positives', 0),\n",
    "                'false_positives': all_metrics[review].get('false_positives', 0),\n",
    "                'false_negatives': all_metrics[review].get('false_negatives', 0),\n",
    "                'true_negatives': all_metrics[review].get('true_negatives', 0)\n",
    "            }\n",
    "        }\n",
    "        \n",
    "        summary_file = output_dir / 'performance_summary.json'\n",
    "        with open(summary_file, 'w') as f:\n",
    "            json.dump(summary, f, indent=2)\n",
    "        \n",
    "        print(f\"  ✓ {review.upper()}: Saved to {output_dir}\")\n",
    "\n",
    "# Save comparative report\n",
    "comparative_report = {\n",
    "    'comparison_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),\n",
    "    'reviews_compared': selected_reviews,\n",
    "    'metrics_comparison': {}\n",
    "}\n",
    "\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        comparative_report['metrics_comparison'][review] = {\n",
    "            k: v for k, v in all_metrics[review].items() \n",
    "            if isinstance(v, (int, float)) and not isinstance(v, bool)\n",
    "        }\n",
    "\n",
    "comparative_file = Path('../data/output/performance_comparison.json')\n",
    "with open(comparative_file, 'w') as f:\n",
    "    json.dump(comparative_report, f, indent=2)\n",
    "\n",
    "print(f\"\\n✅ Comparative report saved: {comparative_file}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Generate Publication-Ready Figures"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create publication-ready figures\n",
    "print(\"🎨 Creating publication-ready figures...\")\n",
    "\n",
    "# 1. Main performance comparison\n",
    "fig_pub1, ax_pub1 = plt.subplots(figsize=(10, 6))\n",
    "\n",
    "metrics_pub = ['Sensitivity', 'Specificity', 'Precision', 'F1 Score', 'Accuracy']\n",
    "x_pub = np.arange(len(metrics_pub))\n",
    "\n",
    "for i, review in enumerate(selected_reviews):\n",
    "    if review in all_metrics and all_metrics[review]:\n",
    "        m = all_metrics[review]\n",
    "        values = [\n",
    "            m.get('sensitivity', 0),\n",
    "            m.get('specificity', 0),\n",
    "            m.get('precision', 0),\n",
    "            m.get('f1_score', 0),\n",
    "            m.get('accuracy', 0)\n",
    "        ]\n",
    "        \n",
    "        ax_pub1.plot(x_pub, values, 'o-', label=review.upper(), \n",
    "                    linewidth=2, markersize=8)\n",
    "\n",
    "ax_pub1.set_xticks(x_pub)\n",
    "ax_pub1.set_xticklabels(metrics_pub)\n",
    "ax_pub1.set_ylim(0, 1.05)\n",
    "ax_pub1.set_ylabel('Score')\n",
    "ax_pub1.set_title('Screening Performance Comparison', fontsize=14, fontweight='bold')\n",
    "ax_pub1.legend()\n",
    "ax_pub1.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/output/performance_comparison.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# 2. Efficiency gains\n",
    "fig_pub2, (ax_pub2a, ax_pub2b) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "\n",
    "# Speed increase\n",
    "speed_data = []\n",
    "review_labels = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review] and 'speed_increase' in all_metrics[review]:\n",
    "        speed_data.append(all_metrics[review]['speed_increase'])\n",
    "        review_labels.append(review.upper())\n",
    "\n",
    "if speed_data:\n",
    "    bars = ax_pub2a.bar(review_labels, speed_data, color=['#2ecc71', '#3498db'], alpha=0.8)\n",
    "    ax_pub2a.set_ylabel('Speed Increase (x)')\n",
    "    ax_pub2a.set_title('Screening Speed Improvement', fontsize=12, fontweight='bold')\n",
    "    ax_pub2a.grid(True, alpha=0.3, axis='y')\n",
    "    \n",
    "    for bar, speed in zip(bars, speed_data):\n",
    "        height = bar.get_height()\n",
    "        ax_pub2a.text(bar.get_x() + bar.get_width()/2., height + 0.5,\n",
    "                     f'{speed:.1f}x', ha='center', fontweight='bold')\n",
    "\n",
    "# Time saved\n",
    "time_data = []\n",
    "for review in selected_reviews:\n",
    "    if review in all_metrics and all_metrics[review] and 'time_saved_hours' in all_metrics[review]:\n",
    "        time_data.append(all_metrics[review]['time_saved_hours'])\n",
    "\n",
    "if time_data and review_labels:\n",
    "    bars = ax_pub2b.bar(review_labels, time_data, color=['#e74c3c', '#f39c12'], alpha=0.8)\n",
    "    ax_pub2b.set_ylabel('Time Saved (hours)')\n",
    "ax_pub2b.set_title('Time Savings', fontsize=12, fontweight='bold')\n",
    "ax_pub2b.grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "for bar, time_saved in zip(bars, time_data):\n",
    "    height = bar.get_height()\n",
    "    ax_pub2b.text(bar.get_x() + bar.get_width()/2., height + 0.5,\n",
    "                 f'{time_saved:.1f}h', ha='center', fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/output/efficiency_gains.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✅ Publication-ready figures saved to ../data/output/\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "cochrane-screening",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}